# KNN — Distance Metric

기존 8,000장 Pilot / 개발 320장 / 공식 320장.

실행 전 INPUT_DIR을 수정하고 새 커널에서 순서대로 실행. 역사적 재현은 1차 실험 당시 baseline을 사용하며 후속 연구는 현재 선정 모델을 사용한다. 출력은 로컬 새 폴더에만 저장. 원본 Test 및 교수님 Test 미사용. 기존 공개 결과는 README/CSV 참고. 분리한 Notebook은 이번 정리에서 전체 학습 재실행하지 않았다.

In [ ]:
from pathlib import Path
import sys, time, json, shutil, gc, os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn import neighbors as nb
from datetime import datetime
START=Path.cwd().resolve()
FT=next((p for p in (START,*START.parents) if (p/"00_Common"/"prepare.py").is_file()),None)
if FT is None: raise FileNotFoundError("Open this notebook within cloned 07_fine_tuning")
sys.path.insert(0,str(FT/"00_Common"))
from prepare import prepare, metrics, evaluate_followup
INPUT_DIR=Path.home()/"ml_fine_tuning_inputs" # set to approved private input directory
OUTPUT_DIR=Path.home()/"ml_fine_tuning_runs"/("KNN_historical_"+datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
CURRENT_SVC=None # followup SVC: set trusted C2_G2 joblib path
state=prepare("KNN", INPUT_DIR, OUTPUT_DIR, mode="historical",current_svc=CURRENT_SVC)
globals().update(state)
def stats(pred,base,y=y):
    return metrics(y,np.asarray(pred),np.asarray(base))
knn=model; tr=knn._fit_X; ty=knn._y; kbase=base
print("Local output:",root)

In [ ]:
# KNN metric pilot - balanced 500 per class training, 20 per class for each dev set.
# Pilot labels are existing train only; no new validation samples are used in fit.
rngk=np.random.default_rng(42)
kp=np.sort(np.concatenate([rngk.choice(np.flatnonzero(ty==c0),500,replace=False) for c0 in range(16)]))
kd=np.sort(np.concatenate([rngk.choice(np.flatnonzero(y==c0),20,replace=False) for c0 in range(16)]))
ko=np.sort(np.concatenate([rngk.choice(np.flatnonzero(yo==c0),20,replace=False) for c0 in range(16)]))
np.savez(root/'03_KNN/EXP-FT-KNN-001_pilot_indices.npz',training_rows=kp,handmade_validation_rows=kd,official_validation_rows=ko)
xx=tr[kp]; yy=ty[kp]
xxv=np.vstack([Xv[kd],Xo[ko]])
yyv=np.r_[y[kd],yo[ko]]
metric_rows=[]
metric_predictions={}
def p_pilot(pv):
 t=time.perf_counter()
 m=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='minkowski',p=pv,algorithm='brute',n_jobs=2).fit(xx,yy)
 pt=m.predict(xxv)
 dur=time.perf_counter()-t
 metric_predictions[str(pv)]=pt
 ss=stats(pt[:len(kd)],kbase[kd],y=y[kd])
 rec={'p':pv,'scope':'pilot','train_rows':len(kp),'official_n':len(ko),'official_correct':int((pt[len(kd):]==yo[ko]).sum()),'elapsed_batch_s':dur,**ss}
 metric_rows.append(rec)
 pd.DataFrame(metric_rows).to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot.csv',index=False)
 print(pv,rec['correct'],rec['macro_f1'],rec['official_correct'],dur,flush=True)
for pv in [2,1]:
 p_pilot(pv)

In [ ]:
def p_pilot_chunk(pv,lo,hi):
 m=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='minkowski',p=pv,algorithm='brute',n_jobs=2).fit(xx,yy)
 t=time.perf_counter();pred=m.predict(xxv[lo:hi]);sec=time.perf_counter()-t
 print(pv,lo,hi,'seconds',sec,flush=True)
 return pred,sec
d15_1,t15_1=p_pilot_chunk(1.5,0,320)

In [ ]:
d15_2,t15_2=p_pilot_chunk(1.5,320,640)
pt=np.r_[d15_1,d15_2]
metric_predictions['1.5']=pt
metric_rows.append({'p':1.5,'scope':'pilot','train_rows':len(kp),'official_n':len(ko),'official_correct':int((pt[len(kd):]==yo[ko]).sum()),'elapsed_batch_s':t15_1+t15_2,**stats(pt[:len(kd)],kbase[kd],y=y[kd])})
pd.DataFrame(metric_rows).to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot.csv',index=False)
print(metric_rows[-1]['correct'],metric_rows[-1]['macro_f1'],metric_rows[-1]['official_correct'])

In [ ]:
d3_1,t3_1=p_pilot_chunk(3,0,320)

In [ ]:
d3_2,t3_2=p_pilot_chunk(3,320,640)
pt=np.r_[d3_1,d3_2];metric_predictions['3']=pt
metric_rows.append({'p':3,'scope':'pilot','train_rows':len(kp),'official_n':len(ko),'official_correct':int((pt[len(kd):]==yo[ko]).sum()),'elapsed_batch_s':t3_1+t3_2,**stats(pt[:len(kd)],kbase[kd],y=y[kd])})
mp=pd.DataFrame(metric_rows);mp['promote']=(mp.correct>mp.correct.iloc[0])&(mp.macro_f1>mp.macro_f1.iloc[0])
mp.to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot.csv',index=False)
pd.DataFrame({'true':yyv,**metric_predictions}).to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot_predictions.csv',index=False)
print(mp[['p','correct','macro_f1','official_correct','elapsed_batch_s','promote']])